# XGBoost-Based Music Recommendation System

## 1. Introduction

This notebook explores a supervised learning approach to music
artist recommendation using XGBoost.

The objective is to predict user-artist interactions and generate
personalized Top-10 artist recommendations.

The model will be evaluated using Recall@10 and NDCG@10 and
compared against previously implemented popularity-based and
K-Means recommendation systems.

In [829]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


## 2. Data Preparation

### Loading the Dataset

We load the cleaned ListenBrainz listening history dataset
to prepare user-artist interactions for supervised learning.

In [830]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"

df = pd.read_parquet(DATA_PATH)

print(df.shape)
display(df.head())

(9974, 6)


,user,artist,artist_mbids,track,release,listened_at
0,jaymiles,Billie Eilish,[f4abc0b5-3f7a-4eff-8f78-ac078dbce533],CHIHIRO,HIT ME HARD AND SOFT,2026-10-12 12:14:04+00:00
1,jaymiles,Billie Eilish,[f4abc0b5-3f7a-4eff-8f78-ac078dbce533],CHIHIRO,HIT ME HARD AND SOFT,2026-10-12 11:58:03+00:00
2,jaymiles,Playboi Carti,[2baf3276-ed6a-4349-8d2e-f4601e7b2167],Yah Mean,Playboi Carti,2026-10-12 11:54:58+00:00
3,jaymiles,Playboi Carti,"[2baf3276-ed6a-4349-8d2e-f4601e7b2167, 800760d...",Choppa Won't Miss (feat. Young Thug),Die Lit,2026-10-12 11:51:15+00:00
4,jaymiles,Grimes,[7e5a2a59-6d9f-4a17-b7c2-e1eedb7bd222],Genesis,Visions,2026-10-12 11:46:38+00:00


###  Chronological Ordering

Listening events are sorted chronologically for each user.

This ensures that historical interactions can be separated
from future interactions when constructing supervised
training examples.

In [831]:
df["listened_at"] = pd.to_datetime(df["listened_at"], utc=True)

df = df.sort_values(by=["user", "listened_at"]).reset_index(drop=True)

display(df.head())

,user,artist,artist_mbids,track,release,listened_at
0,AsuraDtr,Moji x Sboy,[369e57d5-3faa-4e2c-9ca6-d9d96e8e16a4],LA LUNE À DÉCROCHER,LA LUNE À DÉCROCHER,2026-04-04 19:30:07+00:00
1,AsuraDtr,Ino Casablanca,[2b252c46-81f0-4177-9b0f-7771702b2ef2],FLOCAGE,EXTASIA,2026-04-04 19:32:26+00:00
2,AsuraDtr,Damso,"[71ff695e-896a-4043-aea4-93eadde08d09, 3ec4868...",Τ. CHIALER LIVE,QALF infinity,2026-04-04 19:36:12+00:00
3,AsuraDtr,"ISHA, Limsa d'Aulnay","[785498ee-99f4-4699-8caf-2c742e996559, b789f61...",But en or,Bitume Caviar (vol.1),2026-04-04 19:38:51+00:00
4,AsuraDtr,"Luidji, Ryan Koffi",[d30b235e-0967-44a2-8338-2530806fe525],Monde,Saison 00,2026-04-05 10:21:57+00:00


### Chronological Train-Validation-Test Split

We split each user's listening history into three chronological
subsets:

- Training: 80%
- Validation: 10%
- Testing: 10%

This approach preserves the temporal ordering of listening events
and prevents future interactions from being used as historical data.

In [832]:
def chronological_split(group):

    n = len(group)

    train_end = int(n * 0.8)

    val_end = int(n * 0.9)

    train = group.iloc[:train_end]
    val = group.iloc[train_end:val_end]
    test = group.iloc[val_end:]

    return (train, val, test)

In [833]:
train_parts = []
val_parts = []
test_parts = []

for user, group in df.groupby("user"):

    train, val, test = chronological_split(group)

    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

In [834]:
train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (7979, 6)
Validation: (997, 6)
Test: (998, 6)


### Building Supervised Training Examples

We create an additional chronological split within the training
dataset to construct supervised learning examples.

The first 75% of each user's training interactions are used
as historical data.

The remaining 25% are used to identify future artist discoveries.

This prevents information from future listening events from
leaking into the model's training features.

In [835]:
history_parts = []
future_parts = []

for user, group in train_df.groupby("user"):

    split_idx = int(len(group) * 0.75)

    history_parts.append(group.iloc[:split_idx])
    future_parts.append(group.iloc[split_idx:])

history_df = pd.concat(history_parts, ignore_index=True)
future_df = pd.concat(future_parts,ignore_index=True)


print ("history:", history_df.shape)
print("future:", future_df.shape)



history: (5984, 6)
future: (1995, 6)


In [836]:
print("Train:", train_df.shape)
print("History:", history_df.shape)
print("Future:", future_df.shape)

Train: (7979, 6)
History: (5984, 6)
Future: (1995, 6)


###  Positive Interaction Generation

To train XGBoost, we need to transform listening histories into

supervised learning examples.

A positive interaction represents a new artist discovered by a user

during the future observation period.

For each user, we:

1. Identify artists listened to during the historical period.

2. Identify artists listened to during the future period.

3. Compute the difference between the two sets.

4. Assign a target value of 1 to each newly discovered artist.

In [837]:
positive_interactions = []

for user in history_df["user"].unique():

    history_artists = set(
        history_df.loc[history_df["user"] == user, "artist"]
    )

    future_artists = set(
        future_df.loc[future_df["user"] == user, "artist"]
    )

    new_artists = future_artists - history_artists

    for artist in new_artists:
        positive_interactions.append({
            "user": user,
            "artist": artist,
            "target": 1
        })

positive_df = pd.DataFrame(positive_interactions)

print("Positive interactions:", positive_df.shape)

Positive interactions: (872, 3)


In [838]:
print("Positive interactions:", positive_df.shape)
display(positive_df.head())

Positive interactions: (872, 3)


,user,artist,target
0,AsuraDtr,Rounhaa,1
1,AsuraDtr,Dinos,1
2,AsuraDtr,Trinix,1
3,AsuraDtr,"Trinix, Bonga",1
4,AsuraDtr,Hamza,1


### Candidate Artist Catalog

A candidate catalog defines the set of artists that can be
recommended by the model.

To prevent data leakage, the training catalog is constructed
exclusively from historical listening data.

Artists already listened to by a user will be excluded from
that user's recommendation candidates.

This approach ensures that the recommendation system only
considers artists available in the historical training data.

Artists appearing exclusively in the future period are
considered out-of-catalog and cannot be recommended by
the current model.

In [839]:
artist_catalog = set(history_df["artist"].dropna().unique())

print("Candidate artists :", len(artist_catalog))

Candidate artists : 2128


### Candidate Catalog Coverage

We measure the proportion of positive interactions whose
artists are present in the historical candidate catalog.

This metric helps us understand how many future artist
discoveries can potentially be predicted by our model.

A low coverage indicates that many future discoveries
involve artists absent from the historical catalog.

In [840]:
positive_df["in_catalog"] = positive_df["artist"].isin(artist_catalog)

coverage = positive_df["in_catalog"].mean()

print("Catalog coverage:", round(coverage * 100, 2), "%")

Catalog coverage: 39.68 %


In [841]:
eligible_positive_df = positive_df[
    positive_df["in_catalog"]
].copy()

eligible_positive_df = eligible_positive_df[
    ["user", "artist", "target"]
]

print("Eligible positives:", eligible_positive_df.shape)

display(eligible_positive_df.head())

Eligible positives: (346, 3)


,user,artist,target
4,AsuraDtr,Hamza,1
5,AsuraDtr,Folie's,1
7,AsuraDtr,Theodora,1
10,BKILPATRICK11,Joy Crookes,1
12,BKILPATRICK11,JAŸ-Z,1


### 2.10 Negative Sampling

Negative sampling is used to generate user-artist interactions
that were not observed during the future listening period.

For each user, negative candidates are selected from the
historical artist catalog, excluding:

- Artists already present in the user's listening history.
- Artists listened to during the future observation period.

We initially use a negative-to-positive ratio of 3:1.

Random sampling is performed with a fixed random seed
to ensure reproducibility.

Negative interactions are assigned a target value of 0.

In [842]:
import numpy as np

rng = np.random.default_rng(42)

negative_ratio = 3

In [843]:
negative_interactions = []

for user, group in eligible_positive_df.groupby("user"):

    history_artists = set(
        history_df.loc[history_df["user"] == user, "artist"]
    )

    future_artists = set(
        future_df.loc[future_df["user"] == user, "artist"]
    )

    negative_candidates = (
        artist_catalog - history_artists - future_artists
    )

    n_negatives = min(
        len(negative_candidates),
        len(group) * negative_ratio
    )

    sampled_artists = rng.choice(
        sorted(negative_candidates),
        size=n_negatives,
        replace=False
    )

    for artist in sampled_artists:
        negative_interactions.append({
            "user": user,
            "artist": artist,
            "target": 0
        })

In [844]:
negative_df = pd.DataFrame(
    negative_interactions,
    columns=["user", "artist", "target"]
)

print("Negative interactions:", negative_df.shape)

display(negative_df.head())

Negative interactions: (1038, 3)


,user,artist,target
0,AsuraDtr,"David Guetta, Akon",0
1,AsuraDtr,Slowdown,0
2,AsuraDtr,Title Fight,0
3,AsuraDtr,"Paul McCartney, Linda McCartney",0
4,AsuraDtr,Kanye West & André 3000,0


### Building the Supervised Dataset

We combine positive and negative user-artist interactions
into a single supervised learning dataset.

Each observation represents a user-artist pair.

The target variable is binary:

- 1: The user discovered the artist during the future period.
- 0: No interaction with the artist was observed during that period.

This dataset will be used to train an XGBoost classifier.

The next step is to construct numerical features describing
users, artists, and their relationships.

In [845]:
supervised_df = pd.concat([eligible_positive_df, negative_df], 
                          ignore_index=True)

print("Supervised dataset:", supervised_df.shape)

display(supervised_df.head())

Supervised dataset: (1384, 3)


,user,artist,target
0,AsuraDtr,Hamza,1
1,AsuraDtr,Folie's,1
2,AsuraDtr,Theodora,1
3,BKILPATRICK11,Joy Crookes,1
4,BKILPATRICK11,JAŸ-Z,1


In [846]:
print(supervised_df["target"].value_counts())

print(
    supervised_df["target"].value_counts(normalize=True) * 100
)

target
0    1038
1     346
Name: count, dtype: int64
target
0    75.0
1    25.0
Name: proportion, dtype: float64


In [847]:
duplicates = supervised_df.duplicated(
    subset=["user", "artist"]
).sum()

print("Duplicate user-artist pairs:", duplicates)

Duplicate user-artist pairs: 0


## 3. Feature Engineering

Feature engineering transforms historical listening data into
numerical variables that can be used by XGBoost.

All features are computed exclusively from historical data
to prevent information leakage.

### User Activity

User activity measures the total number of listening events
recorded for each user during the historical period.

This feature provides information about the user's
listening activity level. 

In [848]:
user_activity = history_df.groupby("user").size()

print(user_activity.head())

user
AsuraDtr         60
BKILPATRICK11    60
Evan music       60
Giganigga69      60
Gonca12          60
dtype: int64


### 3.2 Mapping User Activity

We map the number of historical listening events to each
user-artist interaction.

This creates our first numerical feature: `user_activity`.

The feature is computed exclusively from historical data
to prevent data leakage.

In [849]:
supervised_df["user_activity"] = supervised_df["user"].map(user_activity)

display(supervised_df.head())

,user,artist,target,user_activity
0,AsuraDtr,Hamza,1,60
1,AsuraDtr,Folie's,1,60
2,AsuraDtr,Theodora,1,60
3,BKILPATRICK11,Joy Crookes,1,60
4,BKILPATRICK11,JAŸ-Z,1,60


In [850]:
print("Missing values:", supervised_df["user_activity"].isna().sum())

print(supervised_df["user_activity"].describe())

Missing values: 0
count    1384.000000
mean       59.953757
std         0.859232
min        44.000000
25%        60.000000
50%        60.000000
75%        60.000000
max        60.000000
Name: user_activity, dtype: float64


In [851]:
print("Missing values:", supervised_df["user_activity"].isna().sum())
print("Standard deviation:", supervised_df["user_activity"].std())
print("Minimum:", supervised_df["user_activity"].min())
print("Maximum:", supervised_df["user_activity"].max())

Missing values: 0
Standard deviation: 0.8592318882374519
Minimum: 44
Maximum: 60


### Artist Popularity

Artist popularity measures the number of unique users who
have listened to a given artist.

Unlike total listening frequency, this feature measures
how widely an artist is listened to across the user population.

Artist popularity is calculated exclusively from historical
listening data to prevent data leakage.

In [852]:
artist_popularity = history_df.groupby("artist")["user"].nunique()

print(artist_popularity.sort_values(ascending=False).head(10))

artist
Kendrick Lamar    24
Travis Scott      23
Don Toliver       23
Kanye West        22
Drake             22
Lil Uzi Vert      21
Playboi Carti     19
J. Cole           18
Frank Ocean       18
Baby Keem         17
Name: user, dtype: int64


###  Mapping Artist Popularity

Artist popularity is mapped to each user-artist pair

in the supervised dataset.

This feature represents the number of distinct users

who listened to the artist during the historical period.

In [853]:
supervised_df["artist_popularity"] = supervised_df["artist"].map(artist_popularity)

display(supervised_df.head())

print("Missing values:", supervised_df["artist_popularity"].isna().sum())

print(supervised_df["artist_popularity"].describe())

,user,artist,target,user_activity,artist_popularity
0,AsuraDtr,Hamza,1,60,4
1,AsuraDtr,Folie's,1,60,1
2,AsuraDtr,Theodora,1,60,2
3,BKILPATRICK11,Joy Crookes,1,60,3
4,BKILPATRICK11,JAŸ-Z,1,60,4


Missing values: 0
count    1384.000000
mean        2.364162
std         3.778814
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        24.000000
Name: artist_popularity, dtype: float64


### User Diversity

User diversity measures the number of unique artists
listened to by each user during the historical period.

Unlike user activity, this feature captures differences
in listening behavior even when users have the same
number of listening events.

The feature is calculated exclusively from historical data.

In [854]:
user_diversity = history_df.groupby("user")["artist"].nunique()

print(user_diversity.describe())

count    100.000000
mean      31.320000
std       12.460751
min        2.000000
25%       23.750000
50%       30.000000
75%       40.000000
max       60.000000
Name: artist, dtype: float64


### Mapping User Diversity

User diversity is mapped to each user-artist interaction
in the supervised dataset.

This feature represents the number of unique artists
listened to by each user during the historical period.

It helps characterize differences in listening behavior
across users.

In [855]:
supervised_df["user_diversity"] = supervised_df["user"].map(user_diversity)

display(supervised_df.head())
print("Missing values:", supervised_df["user_diversity"].isna().sum())

print(supervised_df["user_diversity"].describe())

,user,artist,target,user_activity,artist_popularity,user_diversity
0,AsuraDtr,Hamza,1,60,4,23
1,AsuraDtr,Folie's,1,60,1,23
2,AsuraDtr,Theodora,1,60,2,23
3,BKILPATRICK11,Joy Crookes,1,60,3,33
4,BKILPATRICK11,JAŸ-Z,1,60,4,33


Missing values: 0
count    1384.000000
mean       34.413295
std        12.255168
min         9.000000
25%        26.000000
50%        34.000000
75%        41.000000
max        60.000000
Name: user_diversity, dtype: float64


## 4. User-Artist Affinity Features

User-artist affinity features capture relationships between
users and candidate artists.

We use collaborative filtering to estimate whether an artist
may interest a user based on the listening behavior of
similar users.

### User-Artist Interaction Matrix

We construct a binary user-artist matrix from historical
listening data.

Each row represents a user, and each column represents
an artist.

A value of 1 indicates that the user listened to the artist,
while 0 indicates no observed listening activity.

In [856]:
user_artist_matrix = pd.crosstab(
    history_df["user"],
    history_df["artist"]
)

user_artist_matrix = (user_artist_matrix > 0).astype(int)

print("Matrix shape:", user_artist_matrix.shape)

display(user_artist_matrix.iloc[:5, :10])

Matrix shape: (100, 2128)


artist,$NOT,"$NOT, A$AP Rocky",-M-,.Feast,070 Shake,10cc,1400 Vault,1PLIKATION,1PLIKÉ140,1oneam
user,,,,,,,,,,
AsuraDtr,0,0,0,0,0,0,0,0,0,0
BKILPATRICK11,0,0,0,0,0,0,0,0,0,0
Evan music,0,0,0,0,0,0,0,0,0,0
Giganigga69,0,0,0,0,0,0,0,0,0,0
Gonca12,0,0,0,0,0,0,0,0,0,0


### User Similarity

Cosine similarity measures how similar two users are
based on their historical artist preferences.

A similarity score of 0 indicates no shared artists,
while a score of 1 indicates identical listening profiles
in the binary interaction matrix.

We compute pairwise similarities between all users
using historical listening data only.

In [857]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(user_artist_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (100, 100)


### User Similarity DataFrame

The cosine similarity matrix is converted into a pandas
DataFrame using usernames as row and column labels.

This allows us to identify the most similar users
and inspect their similarity scores.

In [858]:
user_similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_artist_matrix.index,
    columns=user_artist_matrix.index
)

display(user_similarity_df.iloc[:5, :5])

user,AsuraDtr,BKILPATRICK11,Evan music,Giganigga69,Gonca12
user,,,,,
AsuraDtr,1.0,0.000000,0.000000,0.000000,0.000000
BKILPATRICK11,0.0,1.000000,0.000000,0.082572,0.197386
Evan music,0.0,0.000000,1.000000,0.000000,0.142857
Giganigga69,0.0,0.082572,0.000000,1.000000,0.059761
Gonca12,0.0,0.197386,0.142857,0.059761,1.000000


### Nearest Neighbors

We identify the five most similar users based on
cosine similarity.

The target user is excluded from their own neighbors.

These neighbors will later be used to calculate
personalized artist affinity scores.

### 4.7 Generalizing Neighbor Affinity

We define a function to compute the neighbor affinity
score for any user-artist pair.

The function identifies the five nearest neighbors,
excludes the target user, and calculates a
similarity-weighted average of their artist interactions.

Only historical listening data is used.

In [859]:
def compute_neighbor_affinity(user, artist, k=5):

    neighbors = user_similarity_df.loc[user].drop(user)

    top_neighbors = neighbors[neighbors > 0].nlargest(k)

    if top_neighbors.empty:
        return 0.0

    artist_interactions = user_artist_matrix.loc[
        top_neighbors.index, artist
    ]

    affinity = (
        (top_neighbors * artist_interactions).sum()
        / top_neighbors.sum()
    )

    return float(affinity)

### Mapping Neighbor Affinity

We apply the neighbor affinity function to each
user-artist pair in the supervised dataset.

The resulting feature captures personalized
recommendation signals based on similar users'
historical listening behavior.

In [860]:
supervised_df["neighbor_affinity"] = supervised_df.apply(
    lambda row: compute_neighbor_affinity(
        row["user"],
        row["artist"]
    ),
    axis=1
)

display(supervised_df.head())

,user,artist,target,user_activity,artist_popularity,user_diversity,neighbor_affinity
0,AsuraDtr,Hamza,1,60,4,23,0.463026
1,AsuraDtr,Folie's,1,60,1,23,0.000000
2,AsuraDtr,Theodora,1,60,2,23,0.155015
3,BKILPATRICK11,Joy Crookes,1,60,3,33,0.182980
4,BKILPATRICK11,JAŸ-Z,1,60,4,33,0.179797


### Neighbor Affinity Analysis

We analyze the distribution of neighbor affinity scores
across the supervised dataset.

In particular, we measure the proportion of zero scores.

A high proportion of zero scores may indicate sparse
user-artist interactions and limited collaborative
filtering information.

In [861]:
print("Missing values:", supervised_df["neighbor_affinity"].isna().sum())

print(supervised_df["neighbor_affinity"].describe())

zero_percentage = (
    (supervised_df["neighbor_affinity"] == 0).mean() * 100
)

print("Zero affinity percentage:", round(zero_percentage, 2), "%")

Missing values: 0
count    1384.000000
mean        0.039397
std         0.118052
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: neighbor_affinity, dtype: float64
Zero affinity percentage: 86.92 %


### Exploring the Number of Neighbors

We investigate whether increasing the number of nearest
neighbors reduces the sparsity of neighbor affinity scores.

We first compare K=5 with K=20.

A lower proportion of zero affinity scores indicates
greater coverage of candidate artists, but does not
necessarily imply better recommendation performance.

In [862]:
supervised_df["neighbor_affinity_k20"] = supervised_df.apply(
    lambda row: compute_neighbor_affinity(
        row["user"],
        row["artist"],
        k=20
    ),
    axis=1
)

zero_percentage_k20 = (
    (supervised_df["neighbor_affinity_k20"] == 0).mean() * 100
)

print("K=5:", round(zero_percentage, 2), "%")
print("K=20:", round(zero_percentage_k20, 2), "%")

K=5: 86.92 %
K=20: 68.93 %


###  Affinity Analysis by Target Class

We compare neighbor affinity scores between positive
and sampled negative user-artist interactions.

This exploratory analysis helps determine whether
neighbor affinity provides a useful signal for
distinguishing future artist discoveries.

Results are descriptive and do not establish
predictive performance on unseen data.

In [863]:
affinity_comparison = supervised_df.groupby("target")[
    ["neighbor_affinity", "neighbor_affinity_k20"]
].mean()

display(affinity_comparison)

,neighbor_affinity,neighbor_affinity_k20
target,,
0,0.010605,0.012427
1,0.125772,0.114622


## 5. XGBoost Model Training

We prepare the supervised learning dataset by separating
the input features (X) from the target variable (y).

The target indicates whether a user listened to a new
artist during the future training period.

We exclude constant features and identifiers from
the numerical feature matrix.

In [864]:
feature_columns = ["artist_popularity", 
                   "user_diversity",
                   "neighbor_affinity",
                   "neighbor_affinity_k20"
                   ]

X_train = supervised_df[feature_columns]
Y_train = supervised_df["target"]

print ("X_train shape:", X_train.shape)
print ("Y_train shape:", Y_train.shape)

display(X_train.head())
display(Y_train.head())

X_train shape: (1384, 4)
Y_train shape: (1384,)


,artist_popularity,user_diversity,neighbor_affinity,neighbor_affinity_k20
0,4,23,0.463026,0.528414
1,1,23,0.000000,0.101617
2,2,23,0.155015,0.211323
3,3,33,0.182980,0.110434
4,4,33,0.179797,0.123181


0    1
1    1
2    1
3    1
4    1
Name: target, dtype: int64

### Training the XGBoost Classifier

We train an XGBoost classifier to predict future
artist discoveries from user-artist features.

The model uses gradient-boosted decision trees
to learn relationships between historical listening
patterns and future artist interactions.

We start with a simple configuration before
performing hyperparameter tuning.

In [865]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, Y_train)

print("XGBoost training completed!")

XGBoost training completed!


### Training Predictions

We generate prediction scores using the trained
XGBoost classifier.

The model assigns a score to each user-artist pair,
representing its estimated likelihood of belonging
to the positive class under the training distribution.

These training predictions are used for inspection only,
not for evaluating recommendation performance.

In [866]:
train_scores = xgb_model.predict_proba(X_train)[:, 1]

print("Number of predictions:", len(train_scores))

print("First 10 scores:", train_scores[:10])

Number of predictions: 1384
First 10 scores: [0.9805936  0.71259165 0.78654724 0.7260169  0.8060446  0.47584918
 0.29758728 0.4101738  0.8529943  0.24040954]


### Inspecting Model Predictions

We associate XGBoost prediction scores with their
corresponding users, artists, and target labels.

This allows us to inspect how the model ranks
different artists for each user.

These predictions are generated on training data
and are not used to evaluate generalization performance.

In [867]:
train_predictions = supervised_df[
    ["user", "artist", "target"]
].copy()

train_predictions["xgb_score"] = train_scores

display(train_predictions.head(10))

,user,artist,target,xgb_score
0,AsuraDtr,Hamza,1,0.980594
1,AsuraDtr,Folie's,1,0.712592
2,AsuraDtr,Theodora,1,0.786547
3,BKILPATRICK11,Joy Crookes,1,0.726017
4,BKILPATRICK11,JAŸ-Z,1,0.806045
5,BKILPATRICK11,Thundercat,1,0.475849
6,BKILPATRICK11,az chike,1,0.297587
7,Evan music,Damso,1,0.410174
8,Evan music,Lil Baby,1,0.852994
9,Giganigga69,Lana Del Rey,1,0.240410


## 6. Model Validation

We evaluate XGBoost on future listening data that was
not used during model training.

For validation, the complete training period serves
as the historical reference, while the validation
period contains future listening events.

### Identifying New Artist Discoveries

We identify artists listened to during validation
that were not present in each user's training history.

These artists form the ground truth for evaluating
recommendation quality.

In [868]:
train_artists = train_df.groupby("user")["artist"].agg(set)

val_artists = val_df.groupby("user")["artist"].agg(set)

### Validation Ground Truth

We identify new artists discovered by each user
during the validation period.

An artist is considered new if they appear in the
validation listening history but not in the
training listening history.

These discoveries form the ground truth used to
evaluate our recommendation model.

In [869]:
val_new_artists = {}

for user in val_artists.index:

    history = train_artists.get(user, set())

    future = val_artists[user]

    val_new_artists[user] = future - history

### Validation Catalog Coverage

We construct the artist catalog using the training
listening history.

We then measure the proportion of validation
discoveries that belong to this catalog.

Artists absent from the training catalog cannot
be recommended by our current collaborative
filtering approach.

Catalog coverage provides an upper bound on
the achievable recall.

In [870]:
val_catalog = set(train_df["artist"].dropna().unique())

total_discoveries = sum(
    len(artists) for artists in val_new_artists.values()
)

covered_discoveries = sum(
    len(artists & val_catalog)
    for artists in val_new_artists.values()
)

catalog_coverage = covered_discoveries / total_discoveries

print("Catalog size:", len(val_catalog))
print("Total discoveries:", total_discoveries)
print("Covered discoveries:", covered_discoveries)
print("Catalog coverage:", round(catalog_coverage * 100, 2), "%")

Catalog size: 2639
Total discoveries: 414
Covered discoveries: 170
Catalog coverage: 41.06 %


### Validation User-Artist Matrix

We rebuild the user-artist interaction matrix using
the complete training period.

This matrix represents all listening information
available before the validation period.

It will be used to compute user similarities and
personalized affinity features for validation.

In [871]:
val_user_artist_matrix = pd.crosstab(
    train_df["user"],
    train_df["artist"]
)

val_user_artist_matrix = (
    val_user_artist_matrix > 0
).astype(int)

print("Validation matrix shape:", val_user_artist_matrix.shape)

display(val_user_artist_matrix.iloc[:5, :10])

Validation matrix shape: (100, 2639)


artist,$NOT,"$NOT, A$AP Rocky",*67,-M-,.Feast,070 Shake,10cc,13 Block,1400 Vault,1PLIKATION
user,,,,,,,,,,
AsuraDtr,0,0,0,0,0,0,0,0,0,0
BKILPATRICK11,0,0,0,0,0,0,0,0,0,0
Evan music,0,0,0,0,0,0,0,0,0,0
Giganigga69,0,0,0,0,0,0,0,0,0,0
Gonca12,0,0,0,0,0,0,0,0,0,0


###  Validation User Similarity

We compute pairwise cosine similarities between users
using the complete training listening history.

These similarities will be used to identify nearest
neighbors and calculate personalized affinity scores
for validation.

Only historical interactions are used to prevent
information leakage from the validation period.

In [872]:
val_similarity_matrix = cosine_similarity(val_user_artist_matrix)

val_user_similarity_df = pd.DataFrame(
    val_similarity_matrix,
    index=val_user_artist_matrix.index,
    columns=val_user_artist_matrix.index
)

print("Validation similarity shape:", val_user_similarity_df.shape)

display(val_user_similarity_df.iloc[:5, :5])

Validation similarity shape: (100, 100)


user,AsuraDtr,BKILPATRICK11,Evan music,Giganigga69,Gonca12
user,,,,,
AsuraDtr,1.000000,0.000000,0.030773,0.000000,0.000000
BKILPATRICK11,0.000000,1.000000,0.000000,0.088302,0.189258
Evan music,0.030773,0.000000,1.000000,0.000000,0.117851
Giganigga69,0.000000,0.088302,0.000000,1.000000,0.045361
Gonca12,0.000000,0.189258,0.117851,0.045361,1.000000


### Validation Feature Statistics

We recompute artist popularity and user diversity
using the complete training listening history.

Artist popularity measures the number of unique users
who listened to each artist.

User diversity measures the number of distinct artists
listened to by each user.

These features are calculated without using
validation listening data.

In [873]:
val_artist_popularity = train_df.groupby("artist")["user"].nunique()

val_user_diversity = train_df.groupby("user")["artist"].nunique()

print("Artists:", len(val_artist_popularity))
print("Users:", len(val_user_diversity))

display(val_artist_popularity.sort_values(ascending=False).head(10))

Artists: 2639
Users: 100


artist
Drake             29
Kanye West        27
Kendrick Lamar    26
Travis Scott      25
Playboi Carti     25
Don Toliver       24
Lil Uzi Vert      22
Frank Ocean       20
J. Cole           20
Future            19
Name: user, dtype: int64

### Generating Validation Candidates

We generate candidate artists for each user using
the catalog available before the validation period.

Artists already present in a user's training history
are excluded from the recommendation candidates.

We first inspect the candidates for a single user
before generalizing the process to all users.

### 6.8 Building the Validation Candidate Dataset

We create a dataset containing all eligible artist
candidates for a single user.

Each row represents a user-artist pair.

We first add artist popularity and user diversity,
then compute personalized neighbor affinity features.

In [874]:
val_candidates_df = pd.DataFrame({
    "user": user,
    "artist": candidate_artists
})

val_candidates_df["artist_popularity"] = (
    val_candidates_df["artist"].map(val_artist_popularity)
)

val_candidates_df["user_diversity"] = (
    val_candidates_df["user"].map(val_user_diversity)
)

print("Candidate dataset shape:", val_candidates_df.shape)

display(val_candidates_df.head())

Candidate dataset shape: (5993, 4)


,user,artist,artist_popularity,user_diversity
0,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,"""DELUSIONS!"" Robert Pattinson Remix",NaN,77
1,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,"""Profit Margins"" Alchemist",NaN,77
2,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,$NOT,2.0,77
3,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,$Not,NaN,77
4,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,$hyfromdatre,NaN,77


### Computing Validation Neighbor Affinity

We compute personalized neighbor affinity scores
for all candidate artists.

The calculation uses cosine similarities and
historical user-artist interactions.

We compare two neighborhood sizes: K=5 and K=20.

In [875]:
val_neighbors = val_user_similarity_df.loc[user].drop(user)

val_top20 = val_neighbors[val_neighbors > 0].nlargest(20)

val_top5 = val_top20.head(5)

print("K=5 neighbors:", len(val_top5))
print("K=20 neighbors:", len(val_top20))

display(val_top5)

K=5 neighbors: 5
K=20 neighbors: 20


user
brain913          0.131145
varshithkumarg    0.114861
wagnersilva17     0.093922
celeronishe       0.091168
kolalutarg        0.081872
Name: ✵𝕯𝖆𝖒𝖎𝖊𝖓✵, dtype: float64

### Vectorized Neighbor Affinity

We compute neighbor affinity scores for all candidate
artists using matrix operations.

Each neighbor's artist interactions are weighted by
their cosine similarity with the target user.

Vectorization avoids calculating each user-artist
affinity score individually.

In [876]:
neighbor_interactions = val_user_artist_matrix.loc[val_top5.index]

weighted_scores = val_top5 @ neighbor_interactions

affinity_k5 = weighted_scores / val_top5.sum()

print("Number of affinity scores:", len(affinity_k5))

display(affinity_k5.sort_values(ascending=False).head(10))

Number of affinity scores: 2639


artist
Future            1.000000
Drake             0.840396
Lil Baby          0.840396
Tory Lanez        0.822273
Travis Scott      0.816905
Playboi Carti     0.816905
Lil Uzi Vert      0.816905
Kanye West        0.662669
Kendrick Lamar    0.639178
Metro Boomin      0.592990
Name: ✵𝕯𝖆𝖒𝖎𝖊𝖓✵, dtype: float64

In [877]:
neighbor_interactions_k20 = val_user_artist_matrix.loc[val_top20.index]

weighted_scores_k20 = val_top20 @ neighbor_interactions_k20

affinity_k20 = weighted_scores_k20 / val_top20.sum()

print("Number of K=20 affinity scores:", len(affinity_k20))

display(affinity_k20.sort_values(ascending=False).head(10))

Number of K=20 affinity scores: 2639


artist
Travis Scott      0.722908
Future            0.682948
Playboi Carti     0.682091
Lil Uzi Vert      0.681053
Drake             0.679134
Kanye West        0.612944
Don Toliver       0.580489
Gunna             0.558630
Kendrick Lamar    0.526109
Young Thug        0.520041
Name: ✵𝕯𝖆𝖒𝖎𝖊𝖓✵, dtype: float64

### Adding Neighbor Affinity Features

We add the K=5 and K=20 neighbor affinity scores
to the validation candidate dataset.

Each candidate artist receives two personalized
affinity scores based on the target user's
nearest neighbors.

The resulting dataset contains the four features
required by the trained XGBoost model.

In [878]:
val_candidates_df["neighbor_affinity"] = (
    val_candidates_df["artist"].map(affinity_k5)
)

val_candidates_df["neighbor_affinity_k20"] = (
    val_candidates_df["artist"].map(affinity_k20)
)

print("Candidate dataset shape:", val_candidates_df.shape)

display(val_candidates_df.head())

print("Missing values:")
print(val_candidates_df[feature_columns].isna().sum())

Candidate dataset shape: (5993, 6)


,user,artist,artist_popularity,user_diversity,neighbor_affinity,neighbor_affinity_k20
0,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,"""DELUSIONS!"" Robert Pattinson Remix",NaN,77,NaN,NaN
1,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,"""Profit Margins"" Alchemist",NaN,77,NaN,NaN
2,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,$NOT,2.0,77,0.0,0.037317
3,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,$Not,NaN,77,NaN,NaN
4,✵𝕯𝖆𝖒𝖎𝖊𝖓✵,$hyfromdatre,NaN,77,NaN,NaN


Missing values:
artist_popularity        4196
user_diversity              0
neighbor_affinity        4196
neighbor_affinity_k20    4196
dtype: int64


###  Generating Top-10 Recommendations

We rank all eligible candidate artists by their
predicted XGBoost scores.

The ten highest-ranked artists are selected as
recommendations for the target user.

Previously listened artists have already been
excluded from the candidate dataset.

In [879]:
X_val_user = val_candidates_df[feature_columns]

val_candidates_df["xgb_score"] = xgb_model.predict_proba(X_val_user)[:, 1]

top10_xgb = val_candidates_df.nlargest(10, "xgb_score")

display(
    top10_xgb[
        ["artist", "xgb_score"]
    ]
)

,artist,xgb_score
0,"""DELUSIONS!"" Robert Pattinson Remix",0.972722
1,"""Profit Margins"" Alchemist",0.972722
3,$Not,0.972722
4,$hyfromdatre,0.972722
5,"$hyfromdatre, Apex3400",0.972722
6,$uicideboy$,0.972722
7,"$witchblade, lil sleepy",0.972722
8,&TEAM,0.972722
9,"((( O ))), Please Wait",0.972722
11,*Old version,0.972722


### 6.15 Evaluating Recall@10 for a Single User

We evaluate the recommendations by comparing
the top-10 predicted artists with the artists
actually discovered during validation.

Recall@10 measures the proportion of relevant
artists successfully retrieved among the
ten highest-ranked recommendations.

In [880]:
recommended_artists = set(top10_xgb["artist"])

actual_artists = val_new_artists[user]

hits = recommended_artists & actual_artists

recall_at_10 = len(hits) / len(actual_artists)

print("Actual discoveries:", actual_artists)
print("Correct recommendations:", hits)
print("Recall@10:", round(recall_at_10, 4))

Actual discoveries: {'Joshua Baraka, The Compozers', 'Kethan', 'Timi Dre, Ayo Jay', 'Tems', 'Bee Thee Artiste, X.O, AUGUST IV', 'Drake', 'The Kid LAROI', 'Timi Dre', 'Watendawili, Xenia Manasseh', 'Larsimon'}
Correct recommendations: set()
Recall@10: 0.0


### Generalizing User Neighbor Selection

We define a reusable function to identify the
nearest neighbors of any user.

The function uses cosine similarity scores
computed from the training listening history.

It excludes the target user and returns
the K most similar users with positive similarity.

In [881]:
def get_val_neighbors(user, k=5):

    similarities = val_user_similarity_df.loc[user].drop(user)

    top_neighbors = similarities[similarities > 0].nlargest(k)

    return top_neighbors

In [882]:
def compute_val_affinity(user, k=5):

    neighbors = get_val_neighbors(user, k=k)

    if neighbors.empty:
        return pd.Series(0.0, index=val_user_artist_matrix.columns)

    interactions = val_user_artist_matrix.loc[neighbors.index]

    weighted_scores = neighbors @ interactions

    affinity = weighted_scores / neighbors.sum()

    return affinity

### 6.19 Building the XGBoost Recommendation Function

We create a reusable recommendation function that generates
personalized artist recommendations for a given user.

The function will:
- Exclude artists already listened to by the user.
- Compute the required recommendation features.
- Predict artist relevance scores using XGBoost.
- Return the top-ranked artists.

We begin by generating the candidate artist list.

In [883]:
def recommend_artists_xgb(user, top_n=10):

    listened_artists = train_artists[user]

    candidates = sorted(val_catalog - listened_artists)

    print("User:", user)
    print("Candidate artists:", len(candidates))

    return candidates

###  Generating Candidate Features

We construct a feature dataset for all candidate artists.

Each candidate is associated with the same four features
used during XGBoost training.

Features are computed using training history only,
without accessing validation listening events.

###  Adding Neighbor Affinity Features

We calculate personalized affinity scores for each candidate artist.

The scores are based on the listening behavior of the user's
nearest neighbors, weighted by cosine similarity.

Two neighborhood sizes are considered:
- K=5: focuses on the five most similar users.
- K=20: considers a broader group of similar users.

In [884]:
def recommend_artists_xgb(user, top_n=10):

    listened_artists = train_artists[user]

    candidates = sorted(val_catalog - listened_artists)

    candidate_df = pd.DataFrame({
        "user": user,
        "artist": candidates
    })

    candidate_df["artist_popularity"] = (
        candidate_df["artist"].map(val_artist_popularity)
    )

    candidate_df["user_diversity"] = (
        val_user_artist_matrix.loc[user].sum()
    )

    neighbors = val_user_similarity_df.loc[user].drop(user)

    top20 = neighbors[neighbors > 0].nlargest(20)
    top5 = top20.head(5)

    for k, top_neighbors in [(5, top5), (20, top20)]:

        if top_neighbors.empty:
                affinity = pd.Series(0.0, index=val_user_artist_matrix.columns)

        else:
            neighbor_interactions = val_user_artist_matrix.loc[
                top_neighbors.index
         ]

            affinity = (
                top_neighbors @ neighbor_interactions
            ) / top_neighbors.sum()

        column = "neighbor_affinity" if k == 5 else "neighbor_affinity_k20"

        candidate_df[column] = candidate_df["artist"].map(affinity)


    X_candidates = candidate_df[feature_columns]
    candidate_df["xgb_score"] = xgb_model.predict_proba(X_candidates)[:, 1]
    candidate_df = candidate_df.nlargest(top_n, "xgb_score")
    
    return candidate_df

###  Ranking Candidate Artists with XGBoost

We use the trained XGBoost model to assign a prediction
score to each candidate artist.

Candidates are ranked by their predicted scores, and
the 10 highest-ranked artists are selected as recommendations.

Artists already listened to by the user have been excluded.

In [885]:
X_candidates = candidate_df[feature_columns]

candidate_df["xgb_score"] = xgb_model.predict_proba(X_candidates)[:, 1]

top10_recommendations = candidate_df.nlargest(10, "xgb_score")

display(top10_recommendations[["artist", "xgb_score"]])

,artist,xgb_score
3002,Maroon 5,0.930833
652,Bruno Mars,0.924898
2957,Malcolm Todd,0.924898
3806,Rihanna,0.921022
44,3 Doors Down,0.911185
1750,Green Day,0.911185
4940,Yellowcard,0.911185
1782,Gwen Stefani,0.904382
3423,Olivia Dean,0.901056
1018,Daft Punk,0.786054


### Evaluating Recall@10 Across All Users

We evaluate XGBoost recommendations across all validation users.

For each user, we compare the top-10 recommended artists
with the artists actually discovered during validation.

We first verify that the recommendation function works
for multiple users before calculating aggregate metrics.

In [886]:
evaluation_users = list(val_new_artists.keys())

print("Number of evaluation users:", len(evaluation_users))

for user in evaluation_users[:3]:
    recommendations = recommend_artists_xgb(user)

    print(user, ":", len(recommendations), "recommendations")

Number of evaluation users: 100
AsuraDtr : 10 recommendations
BKILPATRICK11 : 10 recommendations
Evan music : 10 recommendations


### Computing Mean Recall@10

We evaluate XGBoost recommendations for all validation users.

For each user, Recall@10 measures the proportion of
new artists discovered during validation that appear
in the top-10 recommendations.

The final score is the average Recall@10 across users
with at least one new artist discovery.

In [887]:
recall_scores = []

for user in evaluation_users:

    actual_artists = val_new_artists[user]

    if len(actual_artists) == 0:
        continue

    recommendations = recommend_artists_xgb(user)

    recommended_artists = set(recommendations["artist"])

    hits = recommended_artists & actual_artists

    recall = len(hits) / len(actual_artists)

    recall_scores.append(recall)

mean_recall = sum(recall_scores) / len(recall_scores)

print("Evaluated users:", len(recall_scores))
print("Mean Recall@10:", round(mean_recall, 4))

Evaluated users: 94
Mean Recall@10: 0.075


### Computing Mean NDCG@10

NDCG@10 evaluates the ranking quality of our recommendations.

Unlike Recall@10, NDCG@10 gives more importance to
relevant artists appearing at higher positions.

We calculate NDCG@10 for each eligible validation user
and report the average score.

In [888]:
import numpy as np

def compute_ndcg_at_10(recommended_artists, actual_artists):

    relevance = [
        1 if artist in actual_artists else 0
        for artist in recommended_artists[:10]
    ]

    dcg = sum(
        rel / np.log2(i + 2)
        for i, rel in enumerate(relevance)
    )

    ideal_hits = min(len(actual_artists), 10)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0.0

In [889]:
ndcg_scores = []

for user in evaluation_users:

    actual_artists = val_new_artists[user]

    if len(actual_artists) == 0:
        continue

    recommendations = recommend_artists_xgb(user)

    recommended_artists = recommendations["artist"].tolist()

    ndcg = compute_ndcg_at_10(recommended_artists, actual_artists)

    ndcg_scores.append(ndcg)

mean_ndcg = sum(ndcg_scores) / len(ndcg_scores)

print("Evaluated users:", len(ndcg_scores))
print("Mean NDCG@10:", round(mean_ndcg, 4))

Evaluated users: 94
Mean NDCG@10: 0.0619


### 6.22 Popularity Baseline

We evaluate a simple popularity-based recommendation model
using the same validation dataset as XGBoost.

Artists are ranked by the number of unique users who
listened to them during the training period.

Previously listened artists are excluded.

This baseline allows us to determine whether XGBoost
outperforms a simple non-personalized recommendation strategy.

In [890]:
def recommend_popular_artists(user, top_n=10):

    listened_artists = train_artists[user]

    candidates = val_artist_popularity.drop(
        labels=list(listened_artists),
        errors="ignore"
    )

    return candidates.nlargest(top_n).index.tolist()

In [891]:
popularity_recall_scores = []
popularity_ndcg_scores = []

for user in evaluation_users:

    actual_artists = val_new_artists[user]

    if len(actual_artists) == 0:
        continue

    recommended_artists = recommend_popular_artists(user)

    hits = set(recommended_artists) & actual_artists

    recall = len(hits) / len(actual_artists)

    ndcg = compute_ndcg_at_10(recommended_artists, actual_artists)

    popularity_recall_scores.append(recall)
    popularity_ndcg_scores.append(ndcg)

print("Evaluated users:", len(popularity_recall_scores))
print("Popularity Recall@10:", round(sum(popularity_recall_scores) / len(popularity_recall_scores), 4))
print("Popularity NDCG@10:", round(sum(popularity_ndcg_scores) / len(popularity_ndcg_scores), 4))

Evaluated users: 94
Popularity Recall@10: 0.087
Popularity NDCG@10: 0.0658


### XGBoost Feature Importance

We analyze the relative importance of the features
used by our trained XGBoost model.

This analysis helps identify which variables contribute
most to the model's predictions.

Feature importance does not necessarily imply causality
or guarantee improved recommendation performance.

In [892]:
feature_importance = pd.Series(
    xgb_model.feature_importances_,
    index=feature_columns
)

display(feature_importance.sort_values(ascending=False))

neighbor_affinity_k20    0.626148
artist_popularity        0.265232
neighbor_affinity        0.058820
user_diversity           0.049801
dtype: float32

### 6.25 Feature Ablation Study

We investigate the contribution of different feature groups
to XGBoost recommendation performance.

By removing selected features and retraining the model,
we can evaluate whether popularity and collaborative
filtering signals improve recommendation quality.

All models will be evaluated using the same validation
dataset and evaluation metrics.

In [893]:
feature_groups = {
    "full_model": [
        "artist_popularity",
        "user_diversity",
        "neighbor_affinity",
        "neighbor_affinity_k20"
    ],

    "without_neighbors": [
        "artist_popularity",
        "user_diversity"
    ],

    "without_popularity": [
        "user_diversity",
        "neighbor_affinity",
        "neighbor_affinity_k20"
    ]
}

for model_name, features in feature_groups.items():
    print(model_name, ":", features)

full_model : ['artist_popularity', 'user_diversity', 'neighbor_affinity', 'neighbor_affinity_k20']
without_neighbors : ['artist_popularity', 'user_diversity']
without_popularity : ['user_diversity', 'neighbor_affinity', 'neighbor_affinity_k20']


In [894]:
from xgboost import XGBClassifier

ablation_models = {}

for model_name, features in feature_groups.items():

    model = XGBClassifier(
        **xgb_model.get_params()
    )

    model.fit(
        X_train[features],
        Y_train
    )

    ablation_models[model_name] = model

    print(model_name, "trained successfully")

full_model trained successfully
without_neighbors trained successfully
without_popularity trained successfully


In [895]:
def recommend_artists_xgb(user, top_n=10, model=None, features=None):
        
    if model is None:
        model = xgb_model

    if features is None:
        features = feature_columns


    listened_artists = train_artists[user]

    candidates = sorted(val_catalog - listened_artists)

    candidate_df = pd.DataFrame({
        "user": user,
        "artist": candidates
    })

    candidate_df["artist_popularity"] = (
        candidate_df["artist"].map(val_artist_popularity)
    )

    candidate_df["user_diversity"] = (
        val_user_artist_matrix.loc[user].sum()
    )

    neighbors = val_user_similarity_df.loc[user].drop(user)

    top20 = neighbors[neighbors > 0].nlargest(20)
    top5 = top20.head(5)

    for k, top_neighbors in [(5, top5), (20, top20)]:

        if top_neighbors.empty:
                affinity = pd.Series(0.0, index=val_user_artist_matrix.columns)

        else:
            neighbor_interactions = val_user_artist_matrix.loc[
                top_neighbors.index
         ]

            affinity = (
                top_neighbors @ neighbor_interactions
            ) / top_neighbors.sum()

        column = "neighbor_affinity" if k == 5 else "neighbor_affinity_k20"

        candidate_df[column] = candidate_df["artist"].map(affinity)


    X_candidates = candidate_df[features]

    candidate_df["xgb_score"] = model.predict_proba(X_candidates)[:, 1]
    
    candidate_df = candidate_df.nlargest(top_n, "xgb_score")
    
    return candidate_df

In [896]:
ablation_results = []

for model_name, features in feature_groups.items():

    model = ablation_models[model_name]

    recall_scores = []
    ndcg_scores = []

    for user in evaluation_users:

        actual_artists = val_new_artists[user]

        if len(actual_artists) == 0:
            continue

        recommendations = recommend_artists_xgb(
            user,
            model=model,
            features=features
        )

        recommended_artists = recommendations["artist"].tolist()

        hits = set(recommended_artists) & actual_artists

        recall = len(hits) / len(actual_artists)

        ndcg = compute_ndcg_at_10(
            recommended_artists,
            actual_artists
        )

        recall_scores.append(recall)
        ndcg_scores.append(ndcg)

    ablation_results.append({
        "model": model_name,
        "recall@10": sum(recall_scores) / len(recall_scores),
        "ndcg@10": sum(ndcg_scores) / len(ndcg_scores)
    })

ablation_results_df = pd.DataFrame(ablation_results)

display(ablation_results_df)

,model,recall@10,ndcg@10
0,full_model,0.074992,0.061949
1,without_neighbors,0.044174,0.036450
2,without_popularity,0.073573,0.061927


## 7. K-Means Recommendation Model

###  Preparing the User-Artist Matrix

We reuse the binary user-artist interaction matrix
built from the training dataset.

Each row represents a user, and each column represents
an artist.

K-Means will use this matrix to group users according
to their listening preferences.

In [897]:
from sklearn.cluster import KMeans

X_kmeans = val_user_artist_matrix.copy()

print("Matrix shape:", X_kmeans.shape)

display(X_kmeans.iloc[:5, :10])

Matrix shape: (100, 2639)


artist,$NOT,"$NOT, A$AP Rocky",*67,-M-,.Feast,070 Shake,10cc,13 Block,1400 Vault,1PLIKATION
user,,,,,,,,,,
AsuraDtr,0,0,0,0,0,0,0,0,0,0
BKILPATRICK11,0,0,0,0,0,0,0,0,0,0
Evan music,0,0,0,0,0,0,0,0,0,0
Giganigga69,0,0,0,0,0,0,0,0,0,0
Gonca12,0,0,0,0,0,0,0,0,0,0


In [898]:
from sklearn.preprocessing import normalize

X_kmeans_normalized = normalize(
    X_kmeans,
    norm="l2"
)

print("Original shape:", X_kmeans.shape)
print("Normalized shape:", X_kmeans_normalized.shape)

Original shape: (100, 2639)
Normalized shape: (100, 2639)


In [899]:
print("Original user norm:", np.linalg.norm(X_kmeans.iloc[0]))
print("Normalized user norm:", np.linalg.norm(X_kmeans_normalized[0]))

Original user norm: 5.744562646538029
Normalized user norm: 1.0


In [900]:
kmeans_normalized = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

normalized_clusters = kmeans_normalized.fit_predict(
    X_kmeans_normalized
)

cluster_distribution_normalized = (
    pd.Series(normalized_clusters)
    .value_counts()
    .sort_index()
)

print("Users per cluster:")
print(cluster_distribution_normalized)

print("\nTotal users:", cluster_distribution_normalized.sum())

Users per cluster:
0    16
1    40
2     8
3     9
4    27
Name: count, dtype: int64

Total users: 100


In [901]:
user_cluster_df = pd.DataFrame({
    "user": X_kmeans.index,
    "cluster": normalized_clusters
})

display(user_cluster_df.head(10))

,user,cluster
0,AsuraDtr,3
1,BKILPATRICK11,0
2,Evan music,1
3,Giganigga69,0
4,Gonca12,0
5,HHTHH,4
6,InnerOtaku,1
7,Kikay,3
8,Kilo.cozel52,4
9,Kraisky,0


###  Computing Artist Popularity Within Clusters

We calculate artist popularity among users belonging
to the same K-Means cluster.

Popularity is measured by the number of unique users
who listened to each artist.

Only training interactions are used to prevent
data leakage from the validation set.

In [902]:
cluster_listens = train_df[
    train_df["user"].isin(same_cluster_users)
]

cluster_artist_popularity = (
    cluster_listens.groupby("artist")["user"]
    .nunique()
    .sort_values(ascending=False)
)

display(cluster_artist_popularity.head(10))

artist
Frank Ocean       6
Mac DeMarco       5
Steve Lacy        4
Arctic Monkeys    4
Kanye West        4
Olivia Dean       4
SZA               4
Daniel Caesar     4
Kendrick Lamar    3
PinkPantheress    3
Name: user, dtype: int64

###  Building the K-Means Recommendation Function

We recommend artists based on their popularity
within each user's K-Means cluster.

Artists already listened to by the target user
are excluded from the recommendations.

The function returns the top 10 unseen artists
ranked by the number of unique listeners in the cluster.

In [903]:
def recommend_artists_kmeans(user, top_n=10):

    cluster_id = user_to_cluster[user]

    cluster_users = user_cluster_df.loc[
        user_cluster_df["cluster"] == cluster_id,
        "user"
    ]

    cluster_listens = train_df[
        train_df["user"].isin(cluster_users)
    ]

    artist_popularity = (
        cluster_listens.groupby("artist")["user"]
        .nunique()
        .sort_values(ascending=False)
    )

    listened_artists = train_artists[user]

    recommendations = artist_popularity.drop(
        labels=list(listened_artists),
        errors="ignore"
    )

    return recommendations.head(top_n)

###  Evaluating K-Means Across All Users

We evaluate the normalized K-Means recommender on all
eligible validation users.

We use the same Recall@10 and NDCG@10 metrics
as the XGBoost and popularity baseline models
to ensure a fair comparison.

In [904]:
kmeans_recalls = []

for user in evaluation_users:

    actual = val_new_artists[user]

    if len(actual) == 0:
        continue

    recommendations = recommend_artists_kmeans(user, top_n=10)

    predicted = set(recommendations.index)

    recall = len(predicted & actual) / len(actual)

    kmeans_recalls.append(recall)

print("Evaluated users:", len(kmeans_recalls))
print("Mean Recall@10:", np.mean(kmeans_recalls))

Evaluated users: 94
Mean Recall@10: 0.08726781492738939


### 7.14 Computing NDCG@10 for K-Means

We evaluate the ranking quality of K-Means recommendations
using Normalized Discounted Cumulative Gain (NDCG@10).

Unlike Recall@10, NDCG@10 rewards relevant artists
appearing at higher positions in the recommendation list.

In [905]:
def ndcg_at_k(recommended, actual, k=10):

    dcg = sum(
        1 / np.log2(i + 2)
        for i, artist in enumerate(recommended[:k])
        if artist in actual
    )

    ideal_hits = min(len(actual), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0

In [906]:
print("Evaluated users:", len(kmeans_ndcgs))
print("Mean Recall@10:", np.mean(kmeans_ndcgs))

Evaluated users: 172
Mean Recall@10: 0.03162891981017586


### 8. Model Comparison on the Validation Set

We compare three recommendation approaches using
the same 172 eligible validation users.

The results show that XGBoost achieves the highest
Recall@10 and NDCG@10, followed by normalized K-Means
and the global popularity baseline.

These results suggest that combining artist popularity
with collaborative filtering features improves
recommendation performance.

However, the limited training catalog coverage
remains a major constraint.

In [908]:
model_comparison = pd.DataFrame({
    "Model": [
        "Popularity",
        "K-Means",
        "XGBoost"
    ],
    "Recall@10": [
        0.087,
        0.0872,
        0.0749
    ],
    "NDCG@10": [
        0.0658,
        0.0316,
        0.0619
    ]
})

display(model_comparison.sort_values(
    by="Recall@10",
    ascending=False
))



,Model,Recall@10,NDCG@10
1,K-Means,0.0872,0.0316
0,Popularity,0.0870,0.0658
2,XGBoost,0.0749,0.0619


In [909]:
display(df.shape)

(9974, 6)